# Object-Oriented Programming (OOP)

**Session 06 (2026-10-24) · Priority 1 · Notebook 4 of 6**

OOP is a way of organising code around **objects**: units that bundle data together with the functions that work on that data.

What you'll learn:

1. Why OOP?
2. Class and object
3. The constructor: `__init__`
4. `self`: what happens behind the scenes
5. Attributes: instance vs class
6. Methods
7. Printing objects: `__str__`
8. Putting it together: a bank account
9. Summary and practice

---
## 1. Why OOP?

So far, we've kept data in variables and dictionaries, and written separate functions that work on them:

In [ ]:
# Without OOP: the data and the function are separate
student_1 = {"name": "asha", "marks": [92, 85, 78]}

def average(student):
    return sum(student["marks"]) / len(student["marks"])

print(average(student_1))

This works, but nothing connects `average()` to student dictionaries. You could pass it the wrong dict, misspell `"marks"` in one place, or forget which functions belong with which data. As programs grow, this gets messy.

OOP bundles the **data** (attributes) and the **behaviour** (methods) into one unit, an **object**:

| Term | Meaning | Real-world analogy |
|---|---|---|
| **Class** | a blueprint or template | an architect's plan for a house |
| **Object** (or *instance*) | one thing built from the class | one actual house built from the plan |
| **Attribute** | data stored in an object | the house's colour, its number of rooms |
| **Method** | a function defined inside a class | what the house can do: open the door, switch on the lights |
| **Constructor** (`__init__`) | sets up a new object's attributes | furnishing a newly built house |
| **`self`** | the object a method is working on | "this particular house" |

**You've been using objects all along.** In Python, everything is an object: `"hello"` is an object of class `str`, and `upper()` is one of its methods.

In [ ]:
text = "hello"
print(type(text))          # <class 'str'>: text is an object of class str
print(text.upper())        # upper() is a method of the str class

numbers = [1, 2]
numbers.append(3)          # append() is a method of the list class
print(type(numbers), numbers)

---
## 2. Class and object

**Syntax:**

```python
class ClassName:       # class names use CamelCase by convention
    ...                # the body: attributes and methods

obj = ClassName()      # calling the class creates a new object
```

In [ ]:
class Student:
    pass                    # an empty class for now ("pass" means "nothing here yet")

s1 = Student()              # create an object: an instance of Student
s2 = Student()              # another, separate object

print(type(s1))
print(s1 is s2)             # False: two different objects
print(isinstance(s1, Student))

You can attach attributes to an object with dot notation, but doing it by hand for every object is error-prone. Forget one, and you get an `AttributeError`:

In [ ]:
s1.name = "asha"
s2.name = "ravi"
print(s1.name, s2.name)

s3 = Student()
try:
    print(s3.name)          # nobody set .name on s3
except AttributeError as e:
    print("AttributeError:", e)

That's exactly the problem a **constructor** solves.

---
## 3. The constructor: `__init__`

**What it is:** a special method that Python calls **automatically** every time you create an object. It sets up the object's starting attributes, so every object is complete from the start.

**Behind the scenes:** `Student("asha", 21)` does three things:

1. creates a new, empty object
2. calls `__init__(new_object, "asha", 21)` to fill it in
3. gives you back the finished object

**Syntax:**

```python
class Student:
    def __init__(self, name, age):
        self.name = name     # create an attribute on this object
        self.age = age
```

**Returns:** `__init__` must return `None`. Its job is to set up `self`, not to produce a value. (`Student(...)` itself returns the new object.)

`__init__` is pronounced "dunder init": **d**ouble **under**score. Dunder methods are special methods that Python calls for you.

In [ ]:
class Student:
    def __init__(self, name, age):
        print(f"__init__ called for {name}")
        self.name = name
        self.age = age

s1 = Student("asha", 21)          # __init__ runs automatically
s2 = Student("ravi", 22)

print(s1.name, s1.age)
print(s2.name, s2.age)

`__init__`'s parameters work like any function's parameters. Leave one out and you get a `TypeError`:

In [ ]:
try:
    Student("asha")               # __init__ needs both name and age
except TypeError as e:
    print("TypeError:", e)

Default values work too, just like default function arguments:

In [ ]:
class Student:
    def __init__(self, name, age, dept="CSE"):
        self.name = name
        self.age = age
        self.dept = dept
        self.marks = []            # each student starts with their own empty list

s1 = Student("asha", 21)
s2 = Student("ravi", 22, dept="ECE")
print(s1.dept, s2.dept)
print(s1.marks, s2.marks)

---
## 4. `self`: what happens behind the scenes

**What it is:** `self` is the first parameter of every method. It refers to **the specific object** the method was called on.

**Behind the scenes:** when you write `s1.greet()`, Python turns it into `Student.greet(s1)`. So inside `greet`, `self` **is** `s1`. That's how one method, written once in the class, can work on many different objects.

In [ ]:
class Student:
    def __init__(self, name):
        self.name = name

    def greet(self):
        print(f"Hi, I am {self.name}  (self is the object with id {id(self)})")

s1 = Student("asha")
s2 = Student("ravi")

s1.greet()               # what you write
Student.greet(s1)        # what Python actually does: the same result
s2.greet()

print("id of s1:", id(s1))
print("id of s2:", id(s2))

**Where do attributes live?** Each object keeps its attributes in its own dictionary, `__dict__`. `self.name = name` is really adding a key to that dictionary.

In [ ]:
print(s1.__dict__)
print(s2.__dict__)

**Two common mistakes with `self`**

1. **Forgetting `self` in the method definition.** Python still passes the object in automatically, so the method receives one argument more than it expects:

In [ ]:
class Broken:
    def greet():                  # forgot self
        print("hi")

try:
    Broken().greet()              # Python passes the object: 1 argument given, 0 expected
except TypeError as e:
    print("TypeError:", e)

2. **Forgetting `self.` in front of an attribute.** Without `self.`, you create a **local variable** that disappears when the method ends:

In [ ]:
class Counter:
    def __init__(self):
        count = 0                 # a local variable, NOT an attribute (missing self.)

c = Counter()
try:
    print(c.count)
except AttributeError as e:
    print("AttributeError:", e)

(`self` is only a naming convention; any name would work. Always use `self` so other people can read your code.)

---
## 5. Attributes: instance vs class

| | Instance attribute | Class attribute |
|---|---|---|
| Defined | inside methods, as `self.x = ...` | directly in the class body |
| Belongs to | one object | the class, so it's **shared** by every object |
| Use it for | data that differs per object (name, balance) | data that's the same for all (a constant, a counter) |

In [ ]:
class Student:
    school = "FDE Bootcamp"          # class attribute: shared by every Student
    count = 0                        # how many students have been created

    def __init__(self, name):
        self.name = name             # instance attribute: different for each student
        Student.count += 1           # update the shared counter through the class

s1 = Student("asha")
s2 = Student("ravi")

print(s1.name, "|", s2.name)         # different
print(s1.school, "|", s2.school)     # the same: both read the class attribute
print("students created:", Student.count)

**Behind the scenes:** when you read `s1.school`, Python looks in `s1.__dict__` first. It's not there, so Python looks in the class next and finds it.

That's why **assigning** through an object doesn't change the class attribute. It creates a new instance attribute on that one object, which hides the class one:

In [ ]:
s1.school = "Other School"           # creates an instance attribute on s1 only
print(s1.school)                     # 'Other School': found on s1 itself
print(s2.school)                     # 'FDE Bootcamp': still the class attribute
print(s1.__dict__)

**Gotcha: a mutable class attribute is shared.** A list in the class body is **one** list that every object uses. This is the reference behaviour from the copying lesson:

In [ ]:
class BadStudent:
    marks = []                       # class attribute: ONE list shared by everyone!

    def __init__(self, name):
        self.name = name

a = BadStudent("asha")
b = BadStudent("ravi")
a.marks.append(92)
print(b.marks)                       # [92]: ravi "got" asha's mark

**Fix:** create lists and dicts inside `__init__` with `self.marks = []`, so each object gets its own (as in section 3).

---
## 6. Methods

**What it is:** a function defined inside a class. It always receives `self`, so it can read and change the object's attributes.

A method can:

- **read** attributes: `self.marks`
- **change** attributes: `self.marks.append(...)`
- **return** a value, like any function
- **call other methods**: `self.average()`
- **raise exceptions** when it gets invalid input

In [ ]:
class Student:
    def __init__(self, name):
        self.name = name
        self.marks = []

    def add_mark(self, mark):                 # changes the object
        if not 0 <= mark <= 100:
            raise ValueError(f"mark must be between 0 and 100, got {mark}")
        self.marks.append(mark)

    def average(self):                        # returns a value
        if not self.marks:
            return 0.0
        return sum(self.marks) / len(self.marks)

    def report(self):                         # calls another method
        return f"{self.name}: {len(self.marks)} marks, average {self.average():.1f}"

asha = Student("asha")
asha.add_mark(92)
asha.add_mark(85)
asha.add_mark(78)

print(asha.marks)
print(asha.average())
print(asha.report())

In [ ]:
try:
    asha.add_mark(150)
except ValueError as e:
    print("ValueError:", e)

print(asha.marks)                     # unchanged: the bad mark was rejected

Validating inside the method means the object can **never** end up holding bad data, wherever it's used.

---
## 7. Printing objects: `__str__`

By default, printing an object shows its class and memory address, which isn't very helpful:

In [ ]:
print(asha)

**What it is:** `__str__` is a dunder method that returns the text to show for an object.

**Behind the scenes:** `print(obj)` calls `str(obj)`, which calls `obj.__str__()`. f-strings do the same.

**Returns:** it must return a `str`.

In [ ]:
class Student:
    def __init__(self, name, dept):
        self.name = name
        self.dept = dept

    def __str__(self):
        return f"Student({self.name}, {self.dept})"

s = Student("asha", "CSE")
print(s)                     # print() uses __str__
print(str(s))
print(f"Found: {s}")         # so do f-strings

There's a related method, `__repr__`, for the developer-facing version. It's what you see when an object sits inside a list or is the last line of a cell. You'll meet it in later sessions.

---
## 8. Putting it together: a bank account

This example uses everything from this notebook, plus exceptions from notebook 01:

- a **custom exception** for a specific error
- a **class attribute** shared by every account
- a **constructor** that validates its input
- **methods** that change the object, return values and raise errors
- `__str__` for printing

In [ ]:
class InsufficientFundsError(Exception):
    pass


class BankAccount:
    bank_name = "FDE Bank"                       # class attribute: shared

    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError("opening balance cannot be negative")
        self.owner = owner                       # instance attributes
        self.balance = balance
        self.transactions = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit amount must be positive")
        self.balance += amount
        self.transactions.append(("deposit", amount))
        return self.balance

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("withdrawal amount must be positive")
        if amount > self.balance:
            raise InsufficientFundsError(
                f"cannot withdraw {amount}, the balance is only {self.balance}"
            )
        self.balance -= amount
        self.transactions.append(("withdraw", amount))
        return self.balance

    def statement(self):
        print(f"--- {self.bank_name}: statement for {self.owner} ---")
        for kind, amount in self.transactions:
            print(f"{kind:<10}{amount:>8}")
        print(f"{'balance':<10}{self.balance:>8}")

    def __str__(self):
        return f"{self.owner}'s account, balance {self.balance}"

In [ ]:
account = BankAccount("asha", 1000)
account.deposit(500)
account.withdraw(200)
print(account)

for amount in [5000, -50]:
    try:
        account.withdraw(amount)
    except InsufficientFundsError as e:
        print("InsufficientFundsError:", e)
    except ValueError as e:
        print("ValueError:", e)

account.statement()

Each object has its own data, but they share the class attribute:

In [ ]:
other = BankAccount("ravi")
other.deposit(300)

print(account)
print(other)
print(account.bank_name, "|", other.bank_name)

---
## 9. Summary

| Concept | Code | Remember |
|---|---|---|
| Class | `class Student:` | the blueprint |
| Object | `s = Student("asha")` | calling the class builds one |
| Constructor | `def __init__(self, ...):` | runs automatically and returns `None` |
| `self` | the first parameter of every method | `s.greet()` becomes `Student.greet(s)` |
| Instance attribute | `self.name = name` | one per object |
| Class attribute | `school = "FDE"` in the class body | shared by all objects |
| Method | `def report(self):` | a function that belongs to the class |
| `__str__` | `def __str__(self): return ...` | what `print()` shows |

**Coming later:** inheritance (one class building on another, like `InsufficientFundsError(Exception)`), encapsulation and polymorphism.

## Practice

**1.** Create a `Rectangle` class with `width` and `height`, plus methods `area()` and `perimeter()`. Raise `ValueError` in the constructor if either side isn't positive.

In [ ]:
# Exercise 1

**2.** Add `__str__` to `Rectangle` so that `print(Rectangle(3, 4))` shows `Rectangle(3 x 4)`.

In [ ]:
# Exercise 2

**3.** Create a `Book` class with `title`, `author` and `available=True`, and methods `borrow()` and `return_book()`. `borrow()` should raise an exception if the book is already borrowed.

In [ ]:
# Exercise 3

**4.** Give a class a class attribute that counts how many objects have been created. Create three objects and print the count.

In [ ]:
# Exercise 4

**5.** (Files + OOP) Add a `save(path)` method to `BankAccount` that writes the owner and balance to a text file, handling `OSError`.

In [ ]:
# Exercise 5